In [2]:
import boto3
from botocore.exceptions import ClientError

# Настройка клиента MinIO через boto3
s3 = boto3.client('s3',
                  endpoint_url='http://localhost:9000',
                  aws_access_key_id='admin',
                  aws_secret_access_key='admin123',
                  region_name='us-east-1')  # MinIO требует указания региона

print("✅ Подключение к MinIO настроено успешно!")
print(f"Endpoint: http://localhost:9000")
print(f"Access Key: admin")

# Проверка подключения - список существующих bucket'ов
try:
    buckets = s3.list_buckets()
    print(f"\n📦 Существующие bucket'ы:")
    for bucket in buckets['Buckets']:
        print(f"  - {bucket['Name']} (создан: {bucket['CreationDate']})")
except ClientError as e:
    print(f"❌ Ошибка подключения: {e}")

✅ Подключение к MinIO настроено успешно!
Endpoint: http://localhost:9000
Access Key: admin

📦 Существующие bucket'ы:
  - mybucket (создан: 2025-08-29 11:51:35.818000+00:00)


In [3]:
# Создание нового bucket'а
bucket_name = 'test-bucket'

try:
    s3.create_bucket(Bucket=bucket_name)
    print(f"✅ Bucket '{bucket_name}' создан успешно!")
except ClientError as e:
    if e.response['Error']['Code'] == 'BucketAlreadyOwnedByYou':
        print(f"📦 Bucket '{bucket_name}' уже существует")
    else:
        print(f"❌ Ошибка создания bucket'а: {e}")

# Список всех bucket'ов после создания
buckets = s3.list_buckets()
print(f"\n📦 Все bucket'ы:")
for bucket in buckets['Buckets']:
    print(f"  - {bucket['Name']}")


✅ Bucket 'test-bucket' создан успешно!

📦 Все bucket'ы:
  - mybucket
  - test-bucket


In [4]:
# Загрузка файла в bucket
file_name = 'myfile.txt'
object_name = 'uploaded-file.txt'

try:
    # Проверяем, существует ли файл
    import os
    if os.path.exists(file_name):
        s3.upload_file(file_name, bucket_name, object_name)
        print(f"✅ Файл '{file_name}' загружен как '{object_name}'")
    else:
        # Создаем тестовый файл, если его нет
        with open(file_name, 'w', encoding='utf-8') as f:
            f.write("Это тестовый файл для MinIO!")
        s3.upload_file(file_name, bucket_name, object_name)
        print(f"✅ Создан и загружен файл '{file_name}' как '{object_name}'")
        
except ClientError as e:
    print(f"❌ Ошибка загрузки файла: {e}")

# Альтернативный способ - загрузка строки как объекта
try:
    s3.put_object(
        Bucket=bucket_name,
        Key='text-object.txt',
        Body='Привет из Python! Это текст, загруженный напрямую.',
        ContentType='text/plain'
    )
    print("✅ Текстовый объект загружен напрямую")
except ClientError as e:
    print(f"❌ Ошибка загрузки объекта: {e}")


✅ Файл 'myfile.txt' загружен как 'uploaded-file.txt'
✅ Текстовый объект загружен напрямую


In [5]:
# Просмотр объектов в bucket'е
try:
    response = s3.list_objects_v2(Bucket=bucket_name)
    
    if 'Contents' in response:
        print(f"📁 Объекты в bucket '{bucket_name}':")
        for obj in response['Contents']:
            size_mb = obj['Size'] / 1024 / 1024
            print(f"  📄 {obj['Key']}")
            print(f"     Размер: {obj['Size']} байт ({size_mb:.2f} MB)")
            print(f"     Изменен: {obj['LastModified']}")
            print()
    else:
        print(f"📦 Bucket '{bucket_name}' пуст")
        
except ClientError as e:
    print(f"❌ Ошибка получения списка объектов: {e}")


📁 Объекты в bucket 'test-bucket':
  📄 text-object.txt
     Размер: 85 байт (0.00 MB)
     Изменен: 2025-08-29 11:57:05.631000+00:00

  📄 uploaded-file.txt
     Размер: 4 байт (0.00 MB)
     Изменен: 2025-08-29 11:57:05.597000+00:00



In [6]:
# Скачивание файла из bucket'а
download_file_name = 'downloaded-file.txt'

try:
    s3.download_file(bucket_name, object_name, download_file_name)
    print(f"✅ Файл '{object_name}' скачан как '{download_file_name}'")
    
    # Читаем содержимое скачанного файла
    with open(download_file_name, 'r', encoding='utf-8') as f:
        content = f.read()
        print(f"📄 Содержимое файла: {content}")
        
except ClientError as e:
    print(f"❌ Ошибка скачивания файла: {e}")

# Альтернативный способ - получение объекта напрямую
try:
    response = s3.get_object(Bucket=bucket_name, Key='text-object.txt')
    content = response['Body'].read().decode('utf-8')
    print(f"\n📄 Содержимое 'text-object.txt': {content}")
    
except ClientError as e:
    print(f"❌ Ошибка получения объекта: {e}")


✅ Файл 'uploaded-file.txt' скачан как 'downloaded-file.txt'
📄 Содержимое файла: asaa

📄 Содержимое 'text-object.txt': Привет из Python! Это текст, загруженный напрямую.
